In [1]:
from random import triangular

import pandas as pd

df = pd.read_csv("./data/raw/kg.csv", low_memory=False)

print(df.shape)
print(df.columns)
print(df.head())

(8100498, 12)
Index(['relation', 'display_relation', 'x_index', 'x_id', 'x_type', 'x_name',
       'x_source', 'y_index', 'y_id', 'y_type', 'y_name', 'y_source'],
      dtype='str')
          relation display_relation  x_index  x_id        x_type  x_name  \
0  protein_protein              ppi        0  9796  gene/protein  PHYHIP   
1  protein_protein              ppi        1  7918  gene/protein  GPANK1   
2  protein_protein              ppi        2  8233  gene/protein   ZRSR2   
3  protein_protein              ppi        3  4899  gene/protein    NRF1   
4  protein_protein              ppi        4  5297  gene/protein   PI4KA   

  x_source  y_index   y_id        y_type  y_name y_source  
0     NCBI     8889  56992  gene/protein   KIF15     NCBI  
1     NCBI     2798   9240  gene/protein   PNMA1     NCBI  
2     NCBI     5646  23548  gene/protein   TTC33     NCBI  
3     NCBI    11592  11253  gene/protein  MAN1B1     NCBI  
4     NCBI     2122   8601  gene/protein   RGS20     NCBI  


In [2]:
print(df["relation"].value_counts())

relation
anatomy_protein_present       3036406
drug_drug                     2672628
protein_protein                642150
disease_phenotype_positive     300634
bioprocess_protein             289610
cellcomp_protein               166804
disease_protein                160822
molfunc_protein                139060
drug_effect                    129568
bioprocess_bioprocess          105772
pathway_protein                 85292
disease_disease                 64388
contraindication                61350
drug_protein                    51306
anatomy_protein_absent          39774
phenotype_phenotype             37472
anatomy_anatomy                 28064
molfunc_molfunc                 27148
indication                      18776
cellcomp_cellcomp                9690
phenotype_protein                6660
off-label use                    5136
pathway_pathway                  5070
exposure_disease                 4608
exposure_exposure                4140
exposure_bioprocess              3250
exp

In [5]:
relations = [
    "drug_protein",
    "protein_protein",
    "disease_protein"
]

graph_df = df[df["relation"].isin(relations)]
print(graph_df["relation"].value_counts())

relation
protein_protein    642150
disease_protein    160822
drug_protein        51306
Name: count, dtype: int64


In [6]:
label_df = df[df["relation"] == "indication"]
print(label_df["relation"].value_counts())

relation
indication    18776
Name: count, dtype: int64


In [8]:
print(df.columns.tolist())
print(
    graph_df[
        ['x_id', 'x_type', 'relation', 'y_id', 'y_type']
    ].head(10)
)

['relation', 'display_relation', 'x_index', 'x_id', 'x_type', 'x_name', 'x_source', 'y_index', 'y_id', 'y_type', 'y_name', 'y_source']
    x_id        x_type         relation    y_id        y_type
0   9796  gene/protein  protein_protein   56992  gene/protein
1   7918  gene/protein  protein_protein    9240  gene/protein
2   8233  gene/protein  protein_protein   23548  gene/protein
3   4899  gene/protein  protein_protein   11253  gene/protein
4   5297  gene/protein  protein_protein    8601  gene/protein
5   6564  gene/protein  protein_protein    8933  gene/protein
6   8668  gene/protein  protein_protein   22976  gene/protein
7  10826  gene/protein  protein_protein  345274  gene/protein
8   4489  gene/protein  protein_protein    7157  gene/protein
9   6272  gene/protein  protein_protein   54873  gene/protein


In [9]:
print(graph_df[["x_index", "y_index"]].head())

print("Min index:", min(
    graph_df["x_index"].min(),
    graph_df["y_index"].min()
))

print("Max index:", max(
    graph_df["x_index"].max(),
    graph_df["y_index"].max()
))

   x_index  y_index
0        0     8889
1        1     2798
2        2     5646
3        3    11592
4        4     2122
Min index: 0
Max index: 95149


In [12]:
import torch

edge_index = torch.tensor(
    graph_df[["x_index", "y_index"]].to_numpy().T,
    dtype=torch.long
)
print(edge_index.shape)
print(edge_index)

torch.Size([2, 854278])
tensor([[    0,     1,     2,  ..., 31190, 33606, 30032],
        [ 8889,  2798,  5646,  ...,  5826, 10422, 33802]])


In [16]:
relation_map = dict(zip(relations, [0, 1, 2]))
print(relation_map)

edge_type = torch.tensor(
    graph_df["relation"].map(relation_map).to_numpy(),
    dtype=torch.long
)
print(edge_type.shape)

{'drug_protein': 0, 'protein_protein': 1, 'disease_protein': 2}
torch.Size([854278])


In [20]:
from torch_geometric.data import Data

num_nodes = int(
    max(
        df["x_index"].max(),
        df["y_index"].max()
    ) + 1
)
print(num_nodes)

data = Data(
    edge_index=edge_index,
    edge_type=edge_type,
    num_nodes=num_nodes,
)
print(data)

129375
Data(edge_index=[2, 854278], edge_type=[854278], num_nodes=129375)


In [22]:
indication_df = df[df["relation"] == "indication"]

print(indication_df[
    ["x_index", "x_type", "y_index", "y_type"]
      ].head(10))

        x_index x_type  y_index   y_type
346730    16687   drug    33577  disease
346731    16687   drug    36035  disease
346764    20297   drug    33577  disease
346765    20297   drug    36035  disease
346768    16693   drug    33577  disease
346769    16693   drug    36035  disease
346770    14944   drug    33577  disease
346771    14944   drug    36035  disease
346774    15762   drug    33577  disease
346775    15762   drug    36035  disease


In [23]:
print(indication_df.groupby(["x_type", "y_type"]).size())

x_type   y_type 
disease  drug       9388
drug     disease    9388
dtype: int64


In [25]:
drug_first = indication_df[
    (indication_df["x_type"] == "drug") & (indication_df["y_type"] == "disease")
]
disease_first = indication_df[
    (indication_df["x_type"] == "disease") & (indication_df["y_type"] == "drug")
]

In [26]:
positive_pairs = (
    drug_first[['x_index', 'y_index']].drop_duplicates().to_numpy()
)

print(positive_pairs.shape)

(9388, 2)


In [30]:
from sklearn.model_selection import train_test_split

train_pos, temp_pos = train_test_split(
    positive_pairs,
    test_size=0.2,
    random_state=42
)

val_pos, test_pos = train_test_split(
    temp_pos,
    test_size=0.5,
    random_state=42
)

In [31]:
drug_ids = df.loc[
    df["x_type"] == "drug", "x_index"
].unique()

disease_ids = df.loc[
    df["x_type"] == "disease", "x_index"
].unique()


In [32]:
import numpy as np

positive_set = set(map(tuple, positive_pairs))

def sample_negatives(n, excluded_pairs, rng):
    negatives = set()

    while len(negatives) < n:
        drug = int(rng.choice(drug_ids))
        disease = int(rng.choice(disease_ids))

        pair = (drug, disease)

        if pair not in excluded_pairs:
            negatives.add(pair)

    return np.array(list(negatives), dtype=np.int64)

In [33]:
rng = np.random.default_rng(42)

train_neg = sample_negatives(
    len(train_pos), positive_set, rng
)

excluded = positive_set | set(map(tuple, train_neg))

val_neg = sample_negatives(
    len(val_pos), excluded, rng
)

excluded |= set(map(tuple, val_neg))

test_neg = sample_negatives(
    len(test_pos), excluded , rng
)

In [35]:
print(train_pos.shape)
print(train_neg.shape)
print(val_pos.shape)
print(val_neg.shape)
print(test_pos.shape)
print(test_neg.shape)

(7510, 2)
(7510, 2)
(939, 2)
(939, 2)
(939, 2)
(939, 2)


In [36]:
import torch
import torch.nn as nn
from torch_geometric.nn import RGCNConv

In [37]:
class RGCN(nn.Module):
    def __init__(self, num_nodes, hidden_dim, num_relations):
        super().__init__()

        self.embedding = nn.Embedding(num_nodes, hidden_dim)
        self.conv1 = RGCNConv(hidden_dim, hidden_dim, num_relations)
        self.conv2 = RGCNConv(hidden_dim, hidden_dim, num_relations)

    def forward(self, edge_index, edge_type):
        x = self.embedding.weight

        x = self.conv1(x, edge_index, edge_type)
        x = torch.relu(x)

        x = self.conv2(x, edge_index, edge_type)

        return x

    def decode(self, z, pairs):
        drug = z[pairs[:, 0]]
        disease = z[pairs[:, 1]]

        scores = (drug * disease).sum(dim=1)

        return scores

In [38]:
model = RGCN(
    num_nodes=data.num_nodes,
    hidden_dim=64,
    num_relations=3
)

In [39]:
z = model(data.edge_index, data.edge_type)

print(z.shape)

torch.Size([129375, 64])


In [40]:
pairs = torch.tensor(train_pos, dtype=torch.long)

scores = model.decode(z, pairs)
print(scores.shape)

torch.Size([7510])


In [41]:
train_pairs = np.concatenate([train_pos, train_neg])
train_labels = np.concatenate([np.ones(len(train_pos)), np.zeros(len(train_neg))])
train_pairs = torch.tensor(train_pairs, dtype=torch.long)
train_labels = torch.tensor(train_labels, dtype=torch.float32)

In [42]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
)

criterion = torch.nn.BCEWithLogitsLoss()

In [44]:
epochs = 100

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()

    z = model(data.edge_index, data.edge_type)
    scores = model.decode(z, train_pairs)
    loss = criterion(scores, train_labels)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch {epoch+1}, "
            f"Loss {loss.item():.4f}, "
        )

Epoch 10, Loss 0.1087, 
Epoch 20, Loss 0.0897, 
Epoch 30, Loss 0.0741, 
Epoch 40, Loss 0.0614, 
Epoch 50, Loss 0.0509, 
Epoch 60, Loss 0.0424, 
Epoch 70, Loss 0.0356, 
Epoch 80, Loss 0.0300, 
Epoch 90, Loss 0.0255, 
Epoch 100, Loss 0.0218, 


In [45]:
from sklearn.metrics import roc_auc_score, average_precision_score

def evaluate(model, data, pos_pairs, neg_pairs):
    model.eval()

    pairs = np.concatenate([pos_pairs, neg_pairs])

    labels = np.concatenate([
        np.ones(len(pos_pairs)),
        np.zeros(len(neg_pairs))
    ])

    pairs = torch.tensor(pairs, dtype=torch.long)

    with torch.no_grad():
        z = model(data.edge_index, data.edge_type)
        scores = model.decode(z, pairs)
        probs = torch.sigmoid(scores).cpu().numpy()

    auroc = roc_auc_score(labels, probs)
    auprc = average_precision_score(labels, probs)

    return auroc, auprc

In [46]:
val_auroc, val_auprc = evaluate(
    model, data, val_pos, val_neg
)

test_auroc, test_auprc = evaluate(
    model, data, test_pos, test_neg
)

print(f"Validation AUROC: {val_auroc:.4f}")
print(f"Validation AUPRC: {val_auprc:.4f}")

print(f"Test AUROC: {test_auroc:.4f}")
print(f"Test AUPRC: {test_auprc:.4f}")

Validation AUROC: 0.8246
Validation AUPRC: 0.7976
Test AUROC: 0.8282
Test AUPRC: 0.7961
